In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_squared_error

In [2]:
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

In [3]:
df = pd.read_csv('C:/Users/Ayyan Aftab/Desktop/iv_surface_reconstruct/data/featured_dataset.csv')
df['datetime'] = pd.to_datetime(df['datetime'])
df.head()

,datetime,underlying_price,contract,iv,option_type,strike,hour,minute,day_of_week,date,session_progress,days_to_expiry,moneyness,log_moneyness,dist_from_atm,dist_from_atm_pct,is_ce,strike_rank,iv_neighbor_-2,iv_neighbor_-1,iv_neighbor_+1,iv_neighbor_+2,iv_neighbor_mean,wide_iv_neighbor_mean,iv_lag_1,iv_roll_mean_5,iv_roll_std_5,iv_roll_mean_10,mean_ce_iv,mean_pe_iv
0,2026-01-07 09:15:00,26111.65,NIFTY27JAN2625200CE,0.12662,CE,25200,9,15,2,2026-01-07,0.000000,20.2569,0.965086,-0.035538,911.65,0.034914,1,0.0,NaN,NaN,0.12330,0.11741,0.12330,0.11741,NaN,NaN,NaN,NaN,0.103060,0.143129
1,2026-01-07 09:20:00,26141.40,NIFTY27JAN2625200CE,0.08632,CE,25200,9,20,2,2026-01-07,0.013514,20.2535,0.963988,-0.036676,941.40,0.036012,1,0.0,NaN,NaN,NaN,NaN,NaN,NaN,0.12662,NaN,NaN,NaN,0.098762,0.147267
2,2026-01-07 09:25:00,26139.35,NIFTY27JAN2625200CE,0.09147,CE,25200,9,25,2,2026-01-07,0.027027,20.2500,0.964064,-0.036598,939.35,0.035936,1,0.0,NaN,NaN,NaN,0.09514,NaN,0.09514,0.08632,0.106470,0.028496,NaN,0.091175,0.148739
3,2026-01-07 09:30:00,26128.95,NIFTY27JAN2625200CE,0.10860,CE,25200,9,30,2,2026-01-07,0.040541,20.2465,0.964447,-0.036200,928.95,0.035553,1,0.0,NaN,NaN,0.10842,0.11150,0.10842,0.11150,0.09147,0.101470,0.021932,0.101470,0.103672,0.142787
4,2026-01-07 09:35:00,26131.90,NIFTY27JAN2625200CE,0.10462,CE,25200,9,35,2,2026-01-07,0.054054,20.2431,0.964339,-0.036313,931.90,0.035661,1,0.0,NaN,NaN,0.10538,0.12459,0.10538,0.12459,0.10860,0.103252,0.018259,0.103252,0.104564,0.147425


In [4]:
df['iv'].isna().sum()

np.int64(5279)

In [6]:
dates = sorted(df['datetime'].dt.date.unique())

folds = [
    {'name': 'Fold 1', 'train_dates': dates[:9],  'val_dates': dates[9:11]},
    {'name': 'Fold 2', 'train_dates': dates[:10], 'val_dates': dates[10:12]},
]

folds = [
    {'name': 'Fold 1', 'train_dates': dates[:9],  'val_dates': dates[9:11]},
    {'name': 'Fold 2', 'train_dates': dates[:10], 'val_dates': dates[10:12]},
]

In [7]:
def make_holdout(val_df):
    np.random.seed(42)

    available_rows = val_df[val_df['iv'].notna()].index

    hidden_rows = np.random.choice(available_rows,size=int(0.2*len(available_rows)),replace=False)

    ground_truth = val_df.loc[hidden_rows,'iv'].values

    val_masked = val_df.copy()

    val_masked.loc[hidden_rows,'iv'] = np.nan

    return val_masked,hidden_rows,ground_truth

In [8]:
def eval_pred(actual,predicted):
    valid = ~np.isnan(predicted)

    mse = mean_squared_error(actual[valid],predicted[valid])
    coverage = valid.mean()

    print(f"MSE: {mse:6f}")
    print(f"Coverage: {coverage:6f}")

    return mse,coverage

In [9]:
import xgboost
import lightgbm
import catboost

In [10]:
from xgboost import XGBRegressor

fold_mses, fold_coverages, feature_importances = [],[],[]

drop_cols = ['iv','datetime','contract','option_type','date']

for fold in folds:
    train_df = df[df['datetime'].dt.date.isin(fold['train_dates'])].copy()

    val_df = df[df['datetime'].dt.date.isin(fold['val_dates'])].copy()

    val_masked, hidden_rows, ground_truth = make_holdout(val_df)

    # train data
    train_obs = train_df[train_df['iv'].notna()].copy()

    X_train = train_obs.drop(columns=drop_cols)
    y_train = train_obs['iv']

    # validation data
    X_val = val_masked.loc[hidden_rows].drop(columns=drop_cols)

    xgb_model = XGBRegressor(
        n_estimators= 500,
        learning_rate= 0.05,
        max_depth= 6,
        subsample= 0.8,
        colsample_bytree= 0.8,
        random_state= 2,
    )

    xgb_model.fit(X_train,y_train)
    xgb_pred = xgb_model.predict(X_val)
    
    xgb_mse, xgb_coverage = eval_pred(ground_truth,xgb_pred)

    fold_mses.append(xgb_mse)
    fold_coverages.append(xgb_coverage)
    feature_importances.append(xgb_model.feature_importances_)

print(f"xgb_mse: {np.mean(fold_mses):6f}")
print(f"xgb_coverage: {np.mean(fold_coverages):6f}")
print(f"feature importances: {feature_importances}")

MSE: 0.000082
Coverage: 1.000000
MSE: 0.000031
Coverage: 1.000000
xgb_mse: 0.000057
xgb_coverage: 1.000000
feature importances: [array([4.0105212e-04, 6.4687044e-03, 2.6302380e-04, 3.2407857e-04,
       4.5344996e-04, 4.8808707e-04, 4.0065980e-04, 1.5535585e-03,
       1.2179541e-03, 8.3658070e-04, 1.6398021e-03, 4.3233731e-04,
       9.5767819e-04, 1.1231116e-03, 2.4564939e-03, 3.0685433e-03,
       1.8700886e-03, 6.5499000e-02, 7.4701919e-03, 8.9218700e-04,
       1.9092827e-01, 6.5716275e-04, 7.0952487e-01, 6.8240991e-04,
       3.9073845e-04], dtype=float32), array([4.0197765e-04, 3.3393078e-03, 2.2084982e-04, 2.2894015e-04,
       5.1717670e-04, 4.3905640e-04, 4.4131003e-04, 1.2302756e-03,
       1.3631552e-03, 7.8754098e-04, 1.2207790e-03, 6.0431979e-04,
       9.9516194e-04, 1.3751175e-03, 2.0852732e-03, 3.0206181e-03,
       1.7931768e-03, 5.5677008e-02, 8.6484179e-03, 1.0532588e-03,
       2.4442613e-01, 6.2081829e-04, 6.6856349e-01, 5.3039298e-04,
       4.1644546e-04], dtype

In [11]:
avg_importance = np.mean(feature_importances,axis=0)

importance_df = pd.DataFrame({
    'feature': X_train.columns,
    'importance': avg_importance
})

importance_df = importance_df.sort_values(by='importance',ascending=False)

importance_df

,feature,importance
22,iv_roll_mean_10,0.689044
20,iv_roll_mean_5,0.217677
17,iv_neighbor_mean,0.060588
18,wide_iv_neighbor_mean,0.008059
1,strike,0.004904
15,iv_neighbor_+1,0.003045
14,iv_neighbor_-1,0.002271
16,iv_neighbor_+2,0.001832
10,dist_from_atm_pct,0.001430
7,moneyness,0.001392


In [12]:
selected_features = importance_df[importance_df['importance']>=0.001]['feature'].tolist()

selected_features

['iv_roll_mean_10',
 'iv_roll_mean_5',
 'iv_neighbor_mean',
 'wide_iv_neighbor_mean',
 'strike',
 'iv_neighbor_+1',
 'iv_neighbor_-1',
 'iv_neighbor_+2',
 'dist_from_atm_pct',
 'moneyness',
 'log_moneyness',
 'iv_neighbor_-2']

In [13]:
df_pruned = df[selected_features+['iv','contract','option_type','date','datetime']].copy()
print(df_pruned.shape)
df_pruned.head()

(26325, 17)


,iv_roll_mean_10,iv_roll_mean_5,iv_neighbor_mean,wide_iv_neighbor_mean,strike,iv_neighbor_+1,iv_neighbor_-1,iv_neighbor_+2,dist_from_atm_pct,moneyness,log_moneyness,iv_neighbor_-2,iv,contract,option_type,date,datetime
0,NaN,NaN,0.12330,0.11741,25200,0.12330,NaN,0.11741,0.034914,0.965086,-0.035538,NaN,0.12662,NIFTY27JAN2625200CE,CE,2026-01-07,2026-01-07 09:15:00
1,NaN,NaN,NaN,NaN,25200,NaN,NaN,NaN,0.036012,0.963988,-0.036676,NaN,0.08632,NIFTY27JAN2625200CE,CE,2026-01-07,2026-01-07 09:20:00
2,NaN,0.106470,NaN,0.09514,25200,NaN,NaN,0.09514,0.035936,0.964064,-0.036598,NaN,0.09147,NIFTY27JAN2625200CE,CE,2026-01-07,2026-01-07 09:25:00
3,0.101470,0.101470,0.10842,0.11150,25200,0.10842,NaN,0.11150,0.035553,0.964447,-0.036200,NaN,0.10860,NIFTY27JAN2625200CE,CE,2026-01-07,2026-01-07 09:30:00
4,0.103252,0.103252,0.10538,0.12459,25200,0.10538,NaN,0.12459,0.035661,0.964339,-0.036313,NaN,0.10462,NIFTY27JAN2625200CE,CE,2026-01-07,2026-01-07 09:35:00


In [15]:
df_pruned['iv'].isna().sum()

np.int64(5279)

In [16]:
df_pruned.to_csv('C:/Users/Ayyan Aftab/Desktop/iv_surface_reconstruct/data/df_pruned.csv',index=False)

In [17]:
fold_mses, fold_coverages = [],[]

drop_cols = ['iv','datetime','contract','option_type','date']

for fold in folds:
    train_df = df_pruned[df_pruned['datetime'].dt.date.isin(fold['train_dates'])].copy()

    val_df = df_pruned[df_pruned['datetime'].dt.date.isin(fold['val_dates'])].copy()

    val_masked, hidden_rows, ground_truth = make_holdout(val_df)

    # train data
    train_obs = train_df[train_df['iv'].notna()].copy()

    X_train = train_obs.drop(columns=drop_cols)
    y_train = train_obs['iv']

    # validation data
    X_val = val_masked.loc[hidden_rows].drop(columns=drop_cols)

    xgb_model = XGBRegressor(
        n_estimators= 500,
        learning_rate= 0.05,
        max_depth= 6,
        subsample= 0.8,
        colsample_bytree= 0.8,
        random_state= 2,
    )

    xgb_model.fit(X_train,y_train)
    xgb_pred = xgb_model.predict(X_val)

    xgb_mse, xgb_coverage = eval_pred(ground_truth,xgb_pred)

    fold_mses.append(xgb_mse)
    fold_coverages.append(xgb_coverage)
    
print(f"xgbp_mse: {np.mean(fold_mses):6f}")
print(f"xgbp_coverage: {np.mean(fold_coverages):6f}")

MSE: 0.000081
Coverage: 1.000000
MSE: 0.000027
Coverage: 1.000000
xgbp_mse: 0.000054
xgbp_coverage: 1.000000


In [18]:
from lightgbm import LGBMRegressor

fold_mses, fold_coverages = [],[]

drop_cols = ['iv','datetime','contract','option_type','date']

for fold in folds:
    train_df = df_pruned[df_pruned['datetime'].dt.date.isin(fold['train_dates'])].copy()

    val_df = df_pruned[df_pruned['datetime'].dt.date.isin(fold['val_dates'])].copy()

    val_masked, hidden_rows, ground_truth = make_holdout(val_df)

    # train data
    train_obs = train_df[train_df['iv'].notna()].copy()

    X_train = train_obs.drop(columns=drop_cols)
    y_train = train_obs['iv']

    # validation data
    X_val = val_masked.loc[hidden_rows].drop(columns=drop_cols)

    lgbm_model = LGBMRegressor(
        n_estimators= 500,
        learning_rate= 0.05,
        max_depth= 6,
        subsample= 0.8,
        colsample_bytree= 0.8,
        random_state= 2,
    )

    lgbm_model.fit(X_train,y_train)
    lgbm_pred = lgbm_model.predict(X_val)
    
    lgbm_mse, lgbm_coverage = eval_pred(ground_truth,lgbm_pred)

    fold_mses.append(lgbm_mse)
    fold_coverages.append(lgbm_coverage)

print(f"MSE from both folds: {fold_mses}")
print(f"lgbm_mse: {np.mean(fold_mses):6f}")
print(f"lgbm_coverage: {np.mean(fold_coverages):6f}")

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001507 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2833
[LightGBM] [Info] Number of data points in the train set: 14546, number of used features: 12
[LightGBM] [Info] Start training from score 0.130303
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain,

In [19]:
from catboost import CatBoostRegressor

fold_mses, fold_coverages = [],[]

drop_cols = ['iv','datetime','contract','option_type','date']

for fold in folds:
    train_df = df_pruned[df_pruned['datetime'].dt.date.isin(fold['train_dates'])].copy()

    val_df = df_pruned[df_pruned['datetime'].dt.date.isin(fold['val_dates'])].copy()

    val_masked, hidden_rows, ground_truth = make_holdout(val_df)

    # train data
    train_obs = train_df[train_df['iv'].notna()].copy()

    X_train = train_obs.drop(columns=drop_cols)
    y_train = train_obs['iv']

    # validation data
    X_val = val_masked.loc[hidden_rows].drop(columns=drop_cols)

    cb_model = CatBoostRegressor(
        n_estimators= 500,
        learning_rate= 0.05,
        max_depth= 6,
        subsample= 0.8,
        random_state= 2,
    )
    
    cb_model.fit(X_train,y_train)
    cb_pred = cb_model.predict(X_val)

    cb_mse, cb_coverage = eval_pred(ground_truth,cb_pred)

    fold_mses.append(cb_mse)
    fold_coverages.append(cb_coverage)

print(f"MSE from both folds: {fold_mses}")
print(f"cb_mse: {np.mean(fold_mses):6f}")
print(f"cb_coverage: {np.mean(fold_coverages):6f}")

0:	learn: 0.0286213	total: 131ms	remaining: 1m 5s
1:	learn: 0.0273051	total: 142ms	remaining: 35.4s
2:	learn: 0.0260467	total: 150ms	remaining: 24.9s
3:	learn: 0.0248556	total: 158ms	remaining: 19.6s
4:	learn: 0.0237177	total: 162ms	remaining: 16s
5:	learn: 0.0226615	total: 166ms	remaining: 13.6s
6:	learn: 0.0216635	total: 170ms	remaining: 12s
7:	learn: 0.0206910	total: 173ms	remaining: 10.6s
8:	learn: 0.0197867	total: 176ms	remaining: 9.61s
9:	learn: 0.0189203	total: 179ms	remaining: 8.79s
10:	learn: 0.0180970	total: 182ms	remaining: 8.08s
11:	learn: 0.0172974	total: 184ms	remaining: 7.5s
12:	learn: 0.0165556	total: 187ms	remaining: 6.99s
13:	learn: 0.0158498	total: 189ms	remaining: 6.55s
14:	learn: 0.0151761	total: 191ms	remaining: 6.18s
15:	learn: 0.0145355	total: 193ms	remaining: 5.84s
16:	learn: 0.0139221	total: 195ms	remaining: 5.54s
17:	learn: 0.0133398	total: 197ms	remaining: 5.27s
18:	learn: 0.0127778	total: 199ms	remaining: 5.04s
19:	learn: 0.0122639	total: 201ms	remaining: 4